# Extract Attention Vectors from Llama-3-8B

**Purpose:** Extract Query (Q), Key (K), and Value (V) vectors from transformer attention layers for offline sparse attention experiments.

**What it does:**
- Loads Llama-3-8B model
- Registers forward hooks on first and last attention layers
- Extracts Q, K, V projections before attention computation for each token position
- Saves extracted vectors to JSONL format for later analysis

**Output format:**
- One JSONL line per example
- Contains Q, K, V for all token positions (enables causal masking)
- Includes both first layer (layer 0) and last layer (layer 31)
- Single head (head 0) to keep file size manageable

**Use case:** Enables reproducible offline experiments on real transformer attention patterns without needing to reload the full model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json

# Load model
model_name = "meta-llama/Meta-Llama-3-8B"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Storage for extracted tensors
attention_cache = {}

def hook_fn(module, args, kwargs, output, layer_idx):
    """
    Hook function to capture Q, K, V from attention layer
    """
    # FIX: Get hidden_states from kwargs (HF style) or args (fallback)
    hidden_states = kwargs.get('hidden_states') if kwargs else args[0]

    # [batch, seq_len, hidden_size]
    batch_size, seq_len, _ = hidden_states.shape
    
    # Project to Q, K, V
    Q = module.q_proj(hidden_states)  # [batch, seq_len, num_heads * head_dim]
    K = module.k_proj(hidden_states)  # [batch, seq_len, num_kv_heads * head_dim]
    V = module.v_proj(hidden_states)  # [batch, seq_len, num_kv_heads * head_dim]
    
    # Reshape to separate heads (Reliance on global 'model' variable preserved)
    Q = Q.view(batch_size, seq_len, model.config.num_attention_heads, -1)
    K = K.view(batch_size, seq_len, model.config.num_key_value_heads, -1)
    V = V.view(batch_size, seq_len, model.config.num_key_value_heads, -1)
    
    # Transpose to [batch, heads, seq_len, head_dim]
    Q = Q.transpose(1, 2)
    K = K.transpose(1, 2)
    V = V.transpose(1, 2)
    
    # Store
    attention_cache[layer_idx] = {
        'Q': Q.detach().cpu(),
        'K': K.detach().cpu(),
        'V': V.detach().cpu()
    }

# Register hooks for first and last layers
first_layer_idx = 0
last_layer_idx = len(model.model.layers) - 1

model.model.layers[first_layer_idx].self_attn.register_forward_hook(
    lambda m, a, k, o: hook_fn(m, a, k, o, first_layer_idx),
    with_kwargs=True
)

model.model.layers[last_layer_idx].self_attn.register_forward_hook(
    lambda m, a, k, o: hook_fn(m, a, k, o, last_layer_idx),
    with_kwargs=True
)

# Process each example
with open('longbench_v2_truncated_7k_smart.json', 'r') as f:
    data = json.load(f)

output_file = open('attention_vectors.jsonl', 'w')

for example in data['examples']:
    # Prepare prompt
    prompt = f"Context: {example['context']}\n\nQuestion: {example['question']}\n\nAnswer:"
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    position_ids = torch.arange(inputs.input_ids.shape[1], device=model.device)
    
    # Clear cache
    attention_cache.clear()
    
    # Forward pass (triggers hooks)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract last token's Q and all K, V
    seq_len = inputs.input_ids.shape[1]
    
    result = {
        "example_id": example.get("_id", "unknown"),
        "domain": example["domain"],
        "question": example["question"],
        "answer": example["answer"],
        "sequence_length": seq_len,
        "model_name": model_name,
        "num_layers": len(model.model.layers),
        "num_heads": model.config.num_attention_heads,
        "num_kv_heads": model.config.num_key_value_heads,
        "head_dim": model.config.hidden_size // model.config.num_attention_heads,
        
        "first_layer": {
            "layer_idx": first_layer_idx,
            "Q": attention_cache[first_layer_idx]['Q'][0, :, -1:, :].tolist(),  # [heads, 1, dim]
            "K": attention_cache[first_layer_idx]['K'][0, :, :, :].tolist(),    # [kv_heads, seq, dim]
            "V": attention_cache[first_layer_idx]['V'][0, :, :, :].tolist(),    # [kv_heads, seq, dim]
            "position_ids": position_ids.tolist()
        },
        
        "last_layer": {
            "layer_idx": last_layer_idx,
            "Q": attention_cache[last_layer_idx]['Q'][0, :, -1:, :].tolist(),
            "K": attention_cache[last_layer_idx]['K'][0, :, :, :].tolist(),
            "V": attention_cache[last_layer_idx]['V'][0, :, :, :].tolist(),
            "position_ids": position_ids.tolist()
        }
    }
    
    # Write to file
    output_file.write(json.dumps(result) + '\n')
    print(f"Processed: {example['domain']} - {seq_len} tokens")
    break;

output_file.close()
print("✅ Extraction complete!")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]